In [1]:
import json

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models

from torch.utils.data import DataLoader

from tqdm import tqdm

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [28]:
train_dir = "../datasets/train"

valid_dir = "../datasets/valid"

In [29]:
train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [30]:
train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [31]:
valid_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [32]:
train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    root=valid_dir,
    transform=valid_transform
)

In [33]:
class_names = train_dataset.classes

print(class_names)

print(len(class_names))

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___healthy']
13


In [34]:
with open(
    "../Backend/models/class_names.json",
    "w"
) as f:

    json.dump(class_names, f)

In [35]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False
)

In [36]:
model = models.efficientnet_b0(
    pretrained=True
)

In [37]:
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(class_names)
)
model = model.to(device)

In [38]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [39]:
epochs = 10

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {epoch_loss:.4f}"
    )

100%|██████████| 775/775 [08:12<00:00,  1.58it/s]


Epoch 1/10 Loss: 0.2931


100%|██████████| 775/775 [03:07<00:00,  4.13it/s]


Epoch 2/10 Loss: 0.0361


100%|██████████| 775/775 [03:07<00:00,  4.13it/s]


...

Epoch 8/10 Loss: 0.0068


100%|██████████| 775/775 [03:08<00:00,  4.12it/s]


Epoch 9/10 Loss: 0.0088


100%|██████████| 775/775 [03:05<00:00,  4.19it/s]

Epoch 10/10 Loss: 0.0053


In [40]:
model.eval()

correct = 0

total = 0

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print(
    f"Validation Accuracy: "
    f"{accuracy:.2f}%"
)

Validation Accuracy: 30.91%


In [41]:
torch.save(
    model.state_dict(),
    "../Backend/models/plant_model.pth"
)

Actual: Tomato_Early_blight
Predicted: Tomato_Early_blight
